In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import os, json
from joblib import dump as joblib_dump, load as joblib_load
import tensorflow as tf
from tensorflow.keras import layers, Model
from google.colab import drive
drive.mount('/content/drive')

# ── Çıktı klasörü (modeller + sonuçlar + tahminler buraya kaydedilir) ───────────
OUTPUT_DIR = "/content/drive/MyDrive/femto_rul/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("OUTPUT_DIR:", OUTPUT_DIR)

# ── Kaydetme yardımcıları (model ağırlıkları cell içinde model.save() ile;
#    burada tahmin dizileri ve sonuç tablosu output/ altına yazılır) ────────────
def save_predictions(pred_dict):
    """output/track_b_predictions.npz'i güncelle (mevcut anahtarları koruyup ekler)."""
    p = f"{OUTPUT_DIR}/track_b_predictions.npz"
    d = dict(np.load(p)) if os.path.exists(p) else {}
    d.update({k: np.asarray(v, dtype=float) for k, v in pred_dict.items()})
    np.savez(p, **d)
    print(f"  [save] {len(pred_dict)} tahmin dizisi → track_b_predictions.npz (toplam {len(d)})")

def save_result(row):
    """output/track_b_results.csv'e model sonucu ekle/güncelle (model adına göre tekilleştir)."""
    p = f"{OUTPUT_DIR}/track_b_results.csv"
    if os.path.exists(p):
        df = pd.read_csv(p); df = df[df['model'] != row['model']]
    else:
        df = pd.DataFrame()
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(p, index=False)
    print(f"  [save] sonuç ({row['model']}) → track_b_results.csv")


Mounted at /content/drive
OUTPUT_DIR: /content/drive/MyDrive/femto_rul/output


In [ ]:
# ============================================================================
# FEMTO RUL — DERİN ÖĞRENME (Track B): HI tabanlı GRU & TCN dizi modeli
# Colab'da çalıştır. Lokal GBM referansı: LOBO HI_traj_r=0.56, test PHM=0.19
#
# Kurgu (lokal deneylerden doğrulanmış):
#   - Hedef = HI (bozulma fraksiyonu 0->1), MUTLAK RUL DEĞİL  (well-posed)
#   - Girdi = son W=24 pencerelik öznitelik DİZİSİ (anlık değil trend)
#   - time_progress YOK, sağlıklı-faz filtresi YOK (ikisi de korelasyonu bozuyor)
#   - HI->RUL: fraksiyon yöntemi + LOBO-seçili gamma kalibrasyonu
#   - Değerlendirme: PHM + endpoint_r + yatak-içi traj_r + LOBO
#   - KAYIT: model ağırlıkları + scaler + meta + sonuç + tahmin -> output/
# ============================================================================
import numpy as np, pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import mutual_info_regression
import warnings; warnings.filterwarnings('ignore')

# ----- Yollar (Colab/Drive'a göre düzenle) -----
TRAIN_CSV = "/content/drive/MyDrive/femto_rul/processed_data/features_train.csv"
TEST_CSV  = "/content/drive/MyDrive/femto_rul/processed_data/features_test.csv"

# ----- Sabitler -----
META=['bearing','window_idx','time_s','rul_s','rul_min','t_star_s','cap_s','condition','split']
W=24; CAP=125.0; SEEDS=[0,1,2,7,42]
T8=['h_std','h_rms','h_kurtosis','h_peak','h_stft_band1_mean','h_stft_band0_mean','v_std','v_rms']

def phm_acc(a,p):
    er=100.0*(a-p)/(a+1e-10); ln=np.log(0.5)
    return np.exp(-ln*(er/5.0)) if er<=0 else np.exp(ln*(er/20.0))

# ----- Ön işleme (lokal pipeline ile BİREBİR) -----
def build(df, raw):
    out=[]
    for b,g in df.groupby('bearing'):
        g=g.sort_values('window_idx').copy()
        g['hi']=np.clip((g['time_s']-g['t_star_s'])/g['cap_s'],0,1)   # HI hedefi (label, özellik DEĞİL)
        for c in raw:                                                  # EMA span20 + baseline çıkarma
            e=g[c].ewm(span=20,adjust=False).mean(); g[c]=e-e.iloc[:20].mean()
        for c in T8: g[c+'_trend']=g[c].diff(5).fillna(0)             # trend (diff5)
        out.append(g)
    return pd.concat(out)

tr=pd.read_csv(TRAIN_CSV); te=pd.read_csv(TEST_CSV)
RAW=[c for c in tr.columns if c not in META]
trb=build(tr,RAW); teb=build(te,RAW); FEATS=RAW+[c+'_trend' for c in T8]

# ----- Öznitelik seçimi (SADECE train; variance + corr>0.95 + MI top-25) -----
keep=[c for c in FEATS if np.std(trb[c].values)>=0.01]
mi=dict(zip(keep,mutual_info_regression(trb[keep].values,trb['hi'].values,random_state=42)))
crm=np.corrcoef(trb[keep].values.T); drop=set()
for i in range(len(keep)):
    for j in range(i+1,len(keep)):
        if abs(crm[i,j])>0.95: drop.add(keep[i] if mi[keep[i]]<mi[keep[j]] else keep[j])
TOP=sorted([c for c in keep if c not in drop],key=lambda c:mi[c],reverse=True)[:25]
print(f"Seçilen {len(TOP)} öznitelik. W={W}")

# ----- Dizi oluşturma: (N, W, F) nedensel pencereler -----
def make_seq(g, scaler):
    g=g.sort_values('window_idx'); M=scaler.transform(g[TOP].values); n=len(M)
    X=np.stack([M[[max(0,i-j) for j in range(W-1,-1,-1)]] for i in range(n)])
    return X, g['hi'].values

def build_dataset(df, scaler):
    Xs,ys=[],[]
    for _,g in df.groupby('bearing'):
        X,y=make_seq(g,scaler); Xs.append(X); ys.append(y)
    return np.concatenate(Xs), np.concatenate(ys)

# ============================================================================
# MODELLER — küçük + güçlü regularizasyon (6 eğri var, overfit riski yüksek)
# ============================================================================
def build_gru(F):
    inp=layers.Input((W,F))
    x=layers.GaussianNoise(0.1)(inp)                     # augmentasyon (genellemeyi artırır)
    x=layers.GRU(48, return_sequences=True, dropout=0.2, recurrent_dropout=0.0,
                 kernel_regularizer=regularizers.l2(1e-3))(x)
    x=layers.GRU(24, dropout=0.2, kernel_regularizer=regularizers.l2(1e-3))(x)
    x=layers.Dense(16,activation='relu',kernel_regularizer=regularizers.l2(1e-3))(x)
    x=layers.Dropout(0.2)(x)
    out=layers.Dense(1,activation='sigmoid')(x)          # HI ∈ [0,1]
    m=Model(inp,out); m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='mse',metrics=['mae'])
    return m

def tcn_block(x,f,d):
    sc=layers.Conv1D(f,1,padding='same')(x)
    c=layers.Conv1D(f,3,padding='causal',dilation_rate=d,kernel_regularizer=regularizers.l2(1e-3))(x)
    c=layers.BatchNormalization()(c); c=layers.Activation('relu')(c); c=layers.SpatialDropout1D(0.15)(c)
    c=layers.Conv1D(f,3,padding='causal',dilation_rate=d,kernel_regularizer=regularizers.l2(1e-3))(c)
    c=layers.BatchNormalization()(c); c=layers.Activation('relu')(c); c=layers.SpatialDropout1D(0.15)(c)
    return layers.Activation('relu')(layers.add([sc,c]))

def build_tcn(F):
    inp=layers.Input((W,F)); x=layers.GaussianNoise(0.1)(inp)
    x=tcn_block(x,32,1); x=tcn_block(x,32,2); x=tcn_block(x,32,4)     # dilation 1,2,4 -> ~W=24 kapsar
    x=layers.GlobalAveragePooling1D()(x)
    x=layers.Dense(16,activation='relu',kernel_regularizer=regularizers.l2(1e-3))(x); x=layers.Dropout(0.2)(x)
    out=layers.Dense(1,activation='sigmoid')(x)
    m=Model(inp,out); m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='mse',metrics=['mae'])
    return m

BUILDERS={'GRU':build_gru,'TCN':build_tcn}

def train_one(builder, Xtr, ytr, Xval, yval, seed):
    tf.keras.utils.set_random_seed(seed)
    m=builder(Xtr.shape[-1])
    es=tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=15,restore_best_weights=True)
    rl=tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=7)
    m.fit(Xtr,ytr,validation_data=(Xval,yval),epochs=120,batch_size=128,
          callbacks=[es,rl],verbose=0)
    return m

# ----- HI->RUL (fraksiyon + gamma), EMA yumuşatma -----
def rul_from_hi(hi, times_s, t_star, gamma):
    h=np.clip(hi,1e-6,1)**gamma
    hs=pd.Series(h).ewm(span=20,adjust=False).mean().values; hl=max(hs[-1],1e-3)
    return float(np.clip(((times_s[-1]-t_star)/60.0)*(1-hl)/hl,0,CAP))

# ============================================================================
# LOBO ile gamma seçimi (DL — 1 seed, hızlı) ve final değerlendirme
# ============================================================================
def predict_bearing(models, g, scaler):
    X,_=make_seq(g,scaler)
    return np.clip(np.mean([m.predict(X,verbose=0).ravel() for m in models],axis=0),0,1)

def run_model(name):
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    GAM=[1.0,0.8,0.7,0.6,0.5,0.4,0.3]
    # ---- LOBO gamma seçimi (sadece train; 1 seed) ----
    lobo={gm:[] for gm in GAM}; hir=[]
    for bout in trb.bearing.unique():
        dtr=trb[trb.bearing!=bout]
        sca=RobustScaler().fit(dtr[TOP].values)
        Xtr,ytr=build_dataset(dtr,sca)
        # iç val: dtr'den 1 yatak
        vb=dtr.bearing.unique()[-1]; Xv,yv=build_dataset(dtr[dtr.bearing==vb],sca)
        m=train_one(BUILDERS[name],Xtr,ytr,Xv,yv,seed=0)
        g=trb[trb.bearing==bout].sort_values('window_idx')
        hi=predict_bearing([m],g,sca)
        if np.std(g['hi'])>1e-6: hir.append(np.corrcoef(g['hi'].values,hi)[0,1])
        times=g['time_s'].values; ts=g['t_star_s'].iloc[0]; total=(g['time_s']+g['rul_s']).max(); n=len(g)
        fpt=int(np.searchsorted(times,ts))
        for fr in [0.3,0.5,0.7,0.85,0.95]:
            u=fpt+int((n-1-fpt)*fr)
            if u<=fpt: continue
            trR=np.clip((total-times[u])/60,0,CAP)
            for gm in GAM: lobo[gm].append(phm_acc(trR, rul_from_hi(hi[:u+1],times[:u+1],ts,gm)))
    gstar=max(GAM,key=lambda gm:np.mean(lobo[gm]))
    lobo_hir=float(np.mean(hir))
    print(f"LOBO HI_traj_r ort = {lobo_hir:+.3f}   LOBO gamma* = {gstar}")

    # ---- Final: tüm train, çoklu seed ensemble ----
    sca=RobustScaler().fit(trb[TOP].values)
    Xtr,ytr=build_dataset(trb,sca)
    vb=trb.bearing.unique()[-1]; Xv,yv=build_dataset(trb[trb.bearing==vb],sca)
    models=[train_one(BUILDERS[name],Xtr,ytr,Xv,yv,s) for s in SEEDS]
    # >>> KAYDET: model ağırlıkları (.keras) + scaler + öznitelik meta (output/)
    for i,_m in enumerate(models): _m.save(f"{OUTPUT_DIR}/{name}_seed{SEEDS[i]}.keras")
    joblib_dump(sca, f"{OUTPUT_DIR}/feature_scaler.pkl")
    json.dump({"TOP":TOP,"W":int(W)}, open(f"{OUTPUT_DIR}/feature_meta.json","w"))
    print(f"  [save] {len(models)} {name} modeli + scaler + meta → output/")

    acts,preds,tj=[],[],[]; _pred={}
    print(f"{'bearing':12s} act    pred   acc")
    rows=[]
    for b,g in teb.groupby('bearing'):
        g=g.sort_values('window_idx'); hi=predict_bearing(models,g,sca)
        _pred[f"{name}__{b}"]=np.asarray(hi,float)   # pencere-pencere HI (trajektori için)
        pr=rul_from_hi(hi,g['time_s'].values,g['t_star_s'].iloc[0],gstar)
        act=np.clip((g['time_s']+g['rul_s']).max()/60-g['time_s'].iloc[-1]/60,0,CAP)
        trr=np.corrcoef(g['hi'].values,hi)[0,1] if np.std(g['hi'])>1e-6 else np.nan
        rows.append((b,act,pr,phm_acc(act,pr))); acts.append(act); preds.append(pr); tj.append(trr)
    for b,a,p,ac in sorted(rows,key=lambda x:x[1]): print(f"  {b:12s} {a:5.1f}  {p:5.1f}  {ac:.3f}")
    acts,preds=np.array(acts),np.array(preds)
    _phm=float(np.mean([phm_acc(a,p) for a,p in zip(acts,preds)]))
    _r=float(np.corrcoef(acts,preds)[0,1]); _mae=float(np.mean(np.abs(acts-preds))); _tjr=float(np.nanmean(tj))
    print(f"\n>>> {name} TEST: PHM={_phm:.4f}  endpoint_r={_r:+.3f}  MAE={_mae:.1f}  yatak-içi_traj_r={_tjr:+.3f}")
    # >>> KAYDET: tahminler (.npz) + sonuç satırı (.csv)
    save_predictions(_pred)
    save_result({"model":name,"phm":round(_phm,4),"endpoint_r":round(_r,3),"mae":round(_mae,1),
                 "traj_r":round(_tjr,3),"lobo_traj_r":round(lobo_hir,3),"gamma":gstar})

for name in ['GRU','TCN']:
    run_model(name)

print("\nReferans (lokal GBM): test PHM=0.19, LOBO HI_traj_r=0.56")
print("Track A (geleneksel, dürüst): XGB ~0.11-0.17")


Seçilen 25 öznitelik. W=24

GRU
LOBO HI_traj_r ort = +0.606   LOBO gamma* = 0.7
  [save] 5 GRU modeli + scaler + meta → output/
bearing      act    pred   acc
  Bearing1_4     5.8    8.5  0.002
  Bearing2_7     9.8    1.9  0.061
  Bearing3_3    13.8    5.5  0.125
  Bearing2_6    21.7   44.9  0.000
  Bearing2_4    23.3  125.0  0.000
  Bearing1_6    24.5  125.0  0.000
  Bearing1_5    27.0   12.9  0.163
  Bearing2_5    51.7   29.0  0.219
  Bearing1_3    95.7  125.0  0.014
  Bearing1_7   125.0  125.0  1.000
  Bearing2_3   125.0  122.8  0.941

>>> GRU TEST: PHM=0.2296  endpoint_r=+0.663  MAE=28.4  yatak-içi_traj_r=+0.339
  [save] 11 tahmin dizisi → track_b_predictions.npz (toplam 11)
  [save] sonuç (GRU) → track_b_results.csv

TCN
LOBO HI_traj_r ort = +0.302   LOBO gamma* = 0.6
  [save] 5 TCN modeli + scaler + meta → output/
bearing      act    pred   acc
  Bearing1_4     5.8   11.5  0.000
  Bearing2_7     9.8   45.5  0.000
  Bearing3_3    13.8   17.9  0.018
  Bearing2_6    21.7  125.0  0.0

In [ ]:
# ============================================================================
# FEMTO RUL — HAM SİNYAL + TCN (RQ2: otomatik FE vs manuel FE ablasyonu)
# Colab + GPU. numpy_data/*.npz (ham X) + processed_data/*.csv (HI etiketi) gerekir.
#
# Tutarlılık: HI hedefi, fraksiyon HI->RUL, LOBO-γ, PHM+r+traj_r — feature pipeline ile AYNI.
# Tek fark: girdi = ham titreşim (2560,2), öznitelikler otomatik (TCN) çıkarılıyor.
# Referans: feature-GRU=0.26 / feature-GBM=0.19 / Track A(geleneksel)=0.11 / CALCE≈0.31
# KAYIT: model ağırlıkları + sonuç + tahmin -> output/
# ============================================================================
import numpy as np, pandas as pd, glob, os
import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers
import warnings; warnings.filterwarnings('ignore')

NPY_DIR="/content/drive/MyDrive/femto_rul/numpy_data"; CSV_DIR="/content/drive/MyDrive/femto_rul/processed_data"; CAP=125.0; SEEDS=[0,1,2]
TRAIN_B=['Bearing1_1','Bearing1_2','Bearing2_1','Bearing2_2','Bearing3_1','Bearing3_2']
def phm_acc(a,p):
    er=100.0*(a-p)/(a+1e-10); ln=np.log(0.5)
    return np.exp(-ln*(er/5.0)) if er<=0 else np.exp(ln*(er/20.0))

# ----- meta (HI etiketi + zaman) CSV'den, ham X .npz'den; pencere sırasıyla hizala -----
csv_tr=pd.read_csv(f"{CSV_DIR}/features_train.csv")
csv_te=pd.read_csv(f"{CSV_DIR}/features_test.csv")
META=pd.concat([csv_tr,csv_te])[['bearing','window_idx','time_s','rul_s','t_star_s','cap_s']]

def baseline_standardize(X, n=50):
    Xn=X.copy().astype(np.float32)
    for ch in (0,1):
        s=max(np.std(X[:min(n,len(X)),:,ch]),1e-5); Xn[:,:,ch]/=s
    return Xn

def load_bearing(bname, is_test):
    pre="test" if is_test else "train"
    d=np.load(f"{NPY_DIR}/{pre}_{bname}.npz"); X=d["X"].astype(np.float32)
    m=META[META.bearing==bname].sort_values('window_idx')
    assert len(m)==len(X), f"{bname}: csv{len(m)} != npz{len(X)}"
    hi=np.clip((m['time_s'].values-m['t_star_s'].values)/m['cap_s'].values,0,1).astype(np.float32)
    return baseline_standardize(X), hi, m['time_s'].values, m['t_star_s'].values[0], (m['time_s']+m['rul_s']).max()

TR={b:load_bearing(b,False) for b in TRAIN_B}
TEST_B=[os.path.basename(f).replace('test_','').replace('.npz','') for f in sorted(glob.glob(f"{NPY_DIR}/test_*.npz"))]
TE={b:load_bearing(b,True) for b in TEST_B}
print("Train:",TRAIN_B); print("Test:",TEST_B)
print("Örnek X şekli:",TR['Bearing1_1'][0].shape)

# ============================================================================
# TCN (ham sinyal) — strided conv ile downsample + dilated bloklar. KÜÇÜK & HIZLI.
# ============================================================================
def tcn_block(x,f,d):
    sc=layers.Conv1D(f,1,padding='same')(x)
    c=layers.Conv1D(f,3,padding='same',dilation_rate=d,kernel_regularizer=regularizers.l2(1e-4))(x)
    c=layers.BatchNormalization()(c); c=layers.Activation('relu')(c); c=layers.SpatialDropout1D(0.1)(c)
    c=layers.Conv1D(f,3,padding='same',dilation_rate=d,kernel_regularizer=regularizers.l2(1e-4))(c)
    c=layers.BatchNormalization()(c); c=layers.Activation('relu')(c); c=layers.SpatialDropout1D(0.1)(c)
    return layers.Activation('relu')(layers.add([sc,c]))

def build_raw_tcn():
    inp=layers.Input((2560,2))
    x=layers.Conv1D(16,16,strides=4,padding='same',activation='relu')(inp)   # 2560->640
    x=layers.BatchNormalization()(x)
    x=layers.Conv1D(32,8,strides=4,padding='same',activation='relu')(x)      # 640->160
    x=layers.BatchNormalization()(x)
    x=layers.Conv1D(32,4,strides=2,padding='same',activation='relu')(x)      # 160->80
    x=layers.BatchNormalization()(x)
    x=tcn_block(x,32,1); x=tcn_block(x,32,2); x=tcn_block(x,32,4)
    x=layers.GlobalAveragePooling1D()(x)
    x=layers.Dense(32,activation='relu',kernel_regularizer=regularizers.l2(1e-4))(x); x=layers.Dropout(0.3)(x)
    out=layers.Dense(1,activation='sigmoid')(x)
    m=Model(inp,out); m.compile(tf.keras.optimizers.Adam(1e-3),loss='mse',metrics=['mae'])
    return m

def train(Xtr,ytr,Xv,yv,seed,epochs=60):
    tf.keras.utils.set_random_seed(seed); m=build_raw_tcn()
    es=tf.keras.callbacks.EarlyStopping('val_loss',patience=10,restore_best_weights=True)
    rl=tf.keras.callbacks.ReduceLROnPlateau('val_loss',factor=0.5,patience=5)
    m.fit(Xtr,ytr,validation_data=(Xv,yv),epochs=epochs,batch_size=128,callbacks=[es,rl],verbose=0)
    return m

def stack(bearings,D):
    return np.concatenate([D[b][0] for b in bearings]), np.concatenate([D[b][1] for b in bearings])
def rul_from_hi(hi,times,ts,gamma):
    h=np.clip(hi,1e-6,1)**gamma; hs=pd.Series(h).ewm(span=20,adjust=False).mean().values
    return float(np.clip(((times[-1]-ts)/60.0)*(1-max(hs[-1],1e-3))/max(hs[-1],1e-3),0,CAP))

# ----- LOBO γ seçimi (1 seed, hızlı) -----
GAM=[1.0,0.8,0.7,0.6,0.5,0.4,0.3]; lobo={g:[] for g in GAM}; hir=[]
print("\nLOBO (6 fold, ham TCN)...")
for bout in TRAIN_B:
    Xtr,ytr=stack([b for b in TRAIN_B if b!=bout],TR)
    Xv,yv=TR[TRAIN_B[-1] if bout!=TRAIN_B[-1] else TRAIN_B[0]][:2]
    m=train(Xtr,ytr,Xv,yv,seed=0,epochs=45)
    Xb,hi_true,times,ts,total=TR[bout]
    hi=np.clip(m.predict(Xb,batch_size=256,verbose=0).ravel(),0,1)
    if np.std(hi_true)>1e-6: hir.append(np.corrcoef(hi_true,hi)[0,1])
    n=len(hi); fpt=int(np.searchsorted(times,ts))
    for fr in [0.3,0.5,0.7,0.85,0.95]:
        u=fpt+int((n-1-fpt)*fr)
        if u<=fpt: continue
        trR=np.clip((total-times[u])/60,0,CAP)
        for g in GAM: lobo[g].append(phm_acc(trR,rul_from_hi(hi[:u+1],times[:u+1],ts,g)))
gstar=max(GAM,key=lambda g:np.mean(lobo[g]))
lobo_hir=float(np.mean(hir))
print(f"LOBO HI_traj_r ort = {lobo_hir:+.3f}   LOBO γ* = {gstar}")

# ----- Final: tüm train, 3-seed ensemble -----
Xtr,ytr=stack(TRAIN_B,TR); Xv,yv=TR['Bearing3_2'][:2]
models=[train(Xtr,ytr,Xv,yv,s,epochs=60) for s in SEEDS]
# >>> KAYDET: ham-TCN v1 model ağırlıkları (output/)
for i,mm in enumerate(models): mm.save(f"{OUTPUT_DIR}/rawTCN_v1_seed{SEEDS[i]}.keras")
print(f"  [save] {len(models)} ham-TCN v1 modeli → output/")

acts,preds,tj=[],[],[]; _pred={}
print(f"\n{'bearing':12s} act    pred   acc")
rows=[]
for b in TEST_B:
    Xb,hi_true,times,ts,total=TE[b]
    hi=np.clip(np.mean([mm.predict(Xb,batch_size=256,verbose=0).ravel() for mm in models],axis=0),0,1)
    _pred[f"raw-TCN__{b}"]=np.asarray(hi,float)   # pencere-pencere HI (trajektori için)
    pr=rul_from_hi(hi,times,ts,gstar); act=np.clip(total/60-times[-1]/60,0,CAP)
    trr=np.corrcoef(hi_true,hi)[0,1] if np.std(hi_true)>1e-6 else np.nan
    rows.append((b,act,pr,phm_acc(act,pr))); acts.append(act); preds.append(pr); tj.append(trr)
for b,a,p,ac in sorted(rows,key=lambda x:x[1]): print(f"  {b:12s} {a:5.1f}  {p:5.1f}  {ac:.3f}")
acts,preds=np.array(acts),np.array(preds)
_phm=float(np.mean([phm_acc(a,p) for a,p in zip(acts,preds)]))
_r=float(np.corrcoef(acts,preds)[0,1]); _mae=float(np.mean(np.abs(acts-preds))); _tjr=float(np.nanmean(tj))
print(f"\n>>> HAM-TCN TEST: PHM={_phm:.4f}  endpoint_r={_r:+.3f}  MAE={_mae:.1f}  yatak-içi_traj_r={_tjr:+.3f}")
# >>> KAYDET: tahminler (.npz) + sonuç satırı (.csv)
save_predictions(_pred)
save_result({"model":"raw-TCN","phm":round(_phm,4),"endpoint_r":round(_r,3),"mae":round(_mae,1),
             "traj_r":round(_tjr,3),"lobo_traj_r":round(lobo_hir,3),"gamma":gstar})
print("\nKıyas: feature-GRU=0.26 | feature-GBM=0.19 | Track A=0.11 | CALCE≈0.31")


Train: ['Bearing1_1', 'Bearing1_2', 'Bearing2_1', 'Bearing2_2', 'Bearing3_1', 'Bearing3_2']
Test: ['Bearing1_3', 'Bearing1_4', 'Bearing1_5', 'Bearing1_6', 'Bearing1_7', 'Bearing2_3', 'Bearing2_4', 'Bearing2_5', 'Bearing2_6', 'Bearing2_7', 'Bearing3_3']
Örnek X şekli: (2803, 2560, 2)

LOBO (6 fold, ham TCN)...


LOBO HI_traj_r ort = +0.600   LOBO γ* = 0.4
  [save] 3 ham-TCN v1 modeli → output/

bearing      act    pred   acc
  Bearing1_4     5.8    0.7  0.049
  Bearing2_7     9.8  125.0  0.000
  Bearing3_3    13.8    0.9  0.039
  Bearing2_6    21.7  125.0  0.000
  Bearing2_4    23.3  125.0  0.000
  Bearing1_6    24.5   37.8  0.001
  Bearing1_5    27.0   10.7  0.123
  Bearing2_5    51.7   21.0  0.128
  Bearing1_3    95.7   20.1  0.065
  Bearing1_7   125.0    1.4  0.032
  Bearing2_3   125.0  125.0  1.000

>>> HAM-TCN TEST: PHM=0.1306  endpoint_r=-0.050  MAE=54.3  yatak-içi_traj_r=+0.452
  [save] 11 tahmin dizisi → track_b_predictions.npz (toplam 33)
  [save] sonuç (raw-TCN) → track_b_results.csv

Kıyas: feature-GRU=0.26 | feature-GBM=0.19 | Track A=0.11 | CALCE≈0.31


In [ ]:
# ============================================================================
# HAM SİNYAL + TCN v2 — kalibrasyon düzeltmeli (LOBO traj_r=0.63 potansiyelini kullan)
# Değişiklikler v1'e göre:
#   (1) Per-bearing HI BASELINE ANCHORING — her yatağın kendi sağlıklı başlangıcına göre
#       HI'yı sıfırla; offset hatasını (1_7'yi bozulmuş sanma) düzeltir. Sızıntısız (yatağın
#       kendi ilk pencereleri inference'ta bilinir).
#   (2) 5 seed ensemble, daha çok epoch.
#   (3) Anchored vs raw HI readout yan yana raporlanır.
# numpy_data/*.npz + processed_data/*.csv gerekir. Colab+GPU.
# KAYIT: v2 model ağırlıkları + (anchored) sonuç -> output/ (tez 5.2 dipnotu; v1 ezilmez)
# ============================================================================
import numpy as np, pandas as pd, glob, os
import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers
import warnings; warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

NPY_DIR="/content/drive/MyDrive/femto_rul/numpy_data"; CSV_DIR="/content/drive/MyDrive/femto_rul/processed_data"; CAP=125.0; SEEDS=[0,1,2,7,42]
TRAIN_B=['Bearing1_1','Bearing1_2','Bearing2_1','Bearing2_2','Bearing3_1','Bearing3_2']
def phm_acc(a,p):
    er=100.0*(a-p)/(a+1e-10); ln=np.log(0.5)
    return np.exp(-ln*(er/5.0)) if er<=0 else np.exp(ln*(er/20.0))

csv_tr=pd.read_csv(f"{CSV_DIR}/features_train.csv"); csv_te=pd.read_csv(f"{CSV_DIR}/features_test.csv")
META=pd.concat([csv_tr,csv_te])[['bearing','window_idx','time_s','rul_s','t_star_s','cap_s']]
def baseline_standardize(X,n=50):
    Xn=X.copy().astype(np.float32)
    for ch in (0,1):
        s=max(np.std(X[:min(n,len(X)),:,ch]),1e-5); Xn[:,:,ch]/=s
    return Xn
def load_bearing(bname,is_test):
    pre="test" if is_test else "train"
    d=np.load(f"{NPY_DIR}/{pre}_{bname}.npz"); X=d["X"].astype(np.float32)
    m=META[META.bearing==bname].sort_values('window_idx'); assert len(m)==len(X)
    hi=np.clip((m['time_s'].values-m['t_star_s'].values)/m['cap_s'].values,0,1).astype(np.float32)
    return baseline_standardize(X),hi,m['time_s'].values,m['t_star_s'].values[0],(m['time_s']+m['rul_s']).max()
TR={b:load_bearing(b,False) for b in TRAIN_B}
TEST_B=[os.path.basename(f).replace('test_','').replace('.npz','') for f in sorted(glob.glob(f"{NPY_DIR}/test_*.npz"))]
TE={b:load_bearing(b,True) for b in TEST_B}

def tcn_block(x,f,d):
    sc=layers.Conv1D(f,1,padding='same')(x)
    c=layers.Conv1D(f,3,padding='same',dilation_rate=d,kernel_regularizer=regularizers.l2(1e-4))(x)
    c=layers.BatchNormalization()(c); c=layers.Activation('relu')(c); c=layers.SpatialDropout1D(0.1)(c)
    c=layers.Conv1D(f,3,padding='same',dilation_rate=d,kernel_regularizer=regularizers.l2(1e-4))(c)
    c=layers.BatchNormalization()(c); c=layers.Activation('relu')(c); c=layers.SpatialDropout1D(0.1)(c)
    return layers.Activation('relu')(layers.add([sc,c]))
def build():
    inp=layers.Input((2560,2))
    x=layers.Conv1D(16,16,strides=4,padding='same',activation='relu')(inp); x=layers.BatchNormalization()(x)
    x=layers.Conv1D(32,8,strides=4,padding='same',activation='relu')(x);  x=layers.BatchNormalization()(x)
    x=layers.Conv1D(48,4,strides=2,padding='same',activation='relu')(x);  x=layers.BatchNormalization()(x)
    x=tcn_block(x,48,1); x=tcn_block(x,48,2); x=tcn_block(x,48,4)
    x=layers.GlobalAveragePooling1D()(x)
    x=layers.Dense(32,activation='relu',kernel_regularizer=regularizers.l2(1e-4))(x); x=layers.Dropout(0.3)(x)
    out=layers.Dense(1,activation='sigmoid')(x)
    m=Model(inp,out); m.compile(tf.keras.optimizers.Adam(1e-3),loss='mse',metrics=['mae']); return m
def trn(Xtr,ytr,Xv,yv,seed,epochs):
    tf.keras.utils.set_random_seed(seed); m=build()
    es=tf.keras.callbacks.EarlyStopping('val_loss',patience=12,restore_best_weights=True)
    rl=tf.keras.callbacks.ReduceLROnPlateau('val_loss',factor=0.5,patience=6)
    m.fit(Xtr,ytr,validation_data=(Xv,yv),epochs=epochs,batch_size=128,callbacks=[es,rl],verbose=0); return m
def stack(bs,D): return np.concatenate([D[b][0] for b in bs]),np.concatenate([D[b][1] for b in bs])

def anchor(hi, K=30):
    """per-bearing baseline anchoring: yatağın kendi sağlıklı başına göre sıfırla, [0,1]'e ölçekle."""
    base=np.median(hi[:max(5,min(K,len(hi)//8))])
    return np.clip((hi-base)/(1.0-base+1e-6),0,1)
def rul(hi,times,ts,gamma,use_anchor):
    h=anchor(hi) if use_anchor else hi.copy()
    h=np.clip(h,1e-6,1)**gamma; hs=pd.Series(h).ewm(span=20,adjust=False).mean().values
    return float(np.clip(((times[-1]-ts)/60.0)*(1-max(hs[-1],1e-3))/max(hs[-1],1e-3),0,CAP))

# ---- LOBO γ seçimi (anchored), 1 seed ----
GAM=[1.0,0.8,0.7,0.6,0.5,0.4,0.3]; lobo={g:[] for g in GAM}; hir=[]
print("LOBO (ham TCN v2, anchored)...")
for bout in TRAIN_B:
    Xtr,ytr=stack([b for b in TRAIN_B if b!=bout],TR)
    Xv,yv=TR['Bearing3_2' if bout!='Bearing3_2' else 'Bearing1_1'][:2]
    m=trn(Xtr,ytr,Xv,yv,0,55)
    Xb,hi_true,times,ts,total=TR[bout]
    hi=np.clip(m.predict(Xb,batch_size=256,verbose=0).ravel(),0,1)
    if np.std(hi_true)>1e-6: hir.append(np.corrcoef(hi_true,hi)[0,1])
    n=len(hi); fpt=int(np.searchsorted(times,ts))
    for fr in [0.3,0.5,0.7,0.85,0.95]:
        u=fpt+int((n-1-fpt)*fr)
        if u<=fpt: continue
        trR=np.clip((total-times[u])/60,0,CAP)
        for g in GAM: lobo[g].append(phm_acc(trR,rul(hi[:u+1],times[:u+1],ts,g,True)))
gstar=max(GAM,key=lambda g:np.mean(lobo[g]))
lobo_hir=float(np.mean(hir))
print(f"LOBO HI_traj_r ort = {lobo_hir:+.3f}   LOBO γ* = {gstar}")

# ---- Final: 5 seed ensemble ----
Xtr,ytr=stack(TRAIN_B,TR); Xv,yv=TR['Bearing3_2'][:2]
models=[trn(Xtr,ytr,Xv,yv,s,90) for s in SEEDS]
# >>> KAYDET: ham-TCN v2 model ağırlıkları (output/)
for i,mm in enumerate(models): mm.save(f"{OUTPUT_DIR}/rawTCN_v2_seed{SEEDS[i]}.keras")
print(f"  [save] {len(models)} ham-TCN v2 modeli → output/")

def evalset(use_anchor,g):
    a,p,tj=[],[],[]
    for b in TEST_B:
        Xb,hi_true,times,ts,total=TE[b]
        hi=np.clip(np.mean([mm.predict(Xb,batch_size=256,verbose=0).ravel() for mm in models],axis=0),0,1)
        pr=rul(hi,times,ts,g,use_anchor); act=np.clip(total/60-times[-1]/60,0,CAP)
        a.append(act); p.append(pr); tj.append(np.corrcoef(hi_true,hi)[0,1] if np.std(hi_true)>1e-6 else np.nan)
    a,p=np.array(a),np.array(p)
    return a,p,np.mean([phm_acc(x,y) for x,y in zip(a,p)]),np.corrcoef(a,p)[0,1],np.nanmean(tj)

_v2=None
for lbl,ua in [("ANCHORED",True),("ham (anchor yok)",False)]:
    a,p,phm,r,tjr=evalset(ua,gstar)
    if ua: _v2=dict(phm=phm,r=r,mae=float(np.mean(np.abs(a-p))),tjr=tjr)
    print(f"\n{lbl}: PHM={phm:.4f}  endpoint_r={r:+.3f}  MAE={np.mean(np.abs(a-p)):.1f}  traj_r={tjr:+.3f}")
    if ua:
        print(f"{'bearing':12s} act    pred   acc")
        for b,x,y in sorted(zip(TEST_B,a,p),key=lambda t:t[1]): print(f"  {b:12s} {x:5.1f}  {y:5.1f}  {phm_acc(x,y):.3f}")
# >>> KAYDET: v2 (anchored) sonucu — tez 5.2 dipnotu ("iyileştirmedi")
save_result({"model":"raw-TCN_v2_anchored","phm":round(_v2['phm'],4),"endpoint_r":round(_v2['r'],3),
             "mae":round(_v2['mae'],1),"traj_r":round(_v2['tjr'],3),"lobo_traj_r":round(lobo_hir,3),"gamma":gstar})
print("\nKıyas: feature-GRU=0.26 | feature-GBM=0.19 | ham-TCN v1=0.13 | CALCE≈0.31")


LOBO (ham TCN v2, anchored)...
LOBO HI_traj_r ort = +0.654   LOBO γ* = 0.6
  [save] 5 ham-TCN v2 modeli → output/

ANCHORED: PHM=0.1392  endpoint_r=-0.111  MAE=58.5  traj_r=+0.484
bearing      act    pred   acc
  Bearing1_4     5.8    0.7  0.046
  Bearing2_7     9.8  125.0  0.000
  Bearing3_3    13.8    1.7  0.048
  Bearing2_6    21.7  125.0  0.000
  Bearing2_4    23.3  125.0  0.000
  Bearing1_6    24.5   81.4  0.000
  Bearing1_5    27.0   19.1  0.364
  Bearing2_5    51.7   80.9  0.000
  Bearing1_3    95.7    7.0  0.040
  Bearing1_7   125.0    1.3  0.032
  Bearing2_3   125.0  125.0  1.000

ham (anchor yok): PHM=0.1392  endpoint_r=-0.112  MAE=58.5  traj_r=+0.484
  [save] sonuç (raw-TCN_v2_anchored) → track_b_results.csv

Kıyas: feature-GRU=0.26 | feature-GBM=0.19 | ham-TCN v1=0.13 | CALCE≈0.31


In [3]:
# ============================================================================
# ÇIKTILARI LİSTELE + İNDİR  (Cell 1/2/3 çalıştıktan SONRA)
# output/ altında olması beklenenler:
#   - Model ağırlıkları: GRU_seed*.keras, TCN_seed*.keras, rawTCN_v1_seed*.keras,
#                        rawTCN_v2_seed*.keras  (+ feature_scaler.pkl, feature_meta.json)
#   - track_b_results.csv   (tüm modellerin PHM/endpoint_r/MAE/traj_r/LOBO/gamma)
#   - track_b_predictions.npz  (pencere-pencere HI; tez trajektori görseli için)
# Artık YENİDEN EĞİTİM YOK — kayıtlar eğitim sırasında yapıldı.
# ============================================================================
import os, pandas as pd
print("output/ içeriği:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {f:32s} {os.path.getsize(os.path.join(OUTPUT_DIR, f))/1024:9.1f} KB")

_rp = f"{OUTPUT_DIR}/track_b_results.csv"
if os.path.exists(_rp):
    print("\n=== track_b_results.csv ===")
    print(pd.read_csv(_rp).to_string(index=False))

# Lokale indir (tez_gorseller.ipynb → experiments/results/ altına koy):
from google.colab import files
for fn in ["track_b_predictions.npz", "track_b_results.csv"]:
    p = f"{OUTPUT_DIR}/{fn}"
    if os.path.exists(p):
        files.download(p)


output/ içeriği:
  GRU_seed0.keras                      238.2 KB
  GRU_seed1.keras                      238.2 KB
  GRU_seed101.keras                    238.2 KB
  GRU_seed11.keras                     238.2 KB
  GRU_seed13.keras                     238.2 KB
  GRU_seed17.keras                     238.2 KB
  GRU_seed2.keras                      238.2 KB
  GRU_seed202.keras                    238.2 KB
  GRU_seed21.keras                     238.2 KB
  GRU_seed3.keras                      238.2 KB
  GRU_seed33.keras                     238.2 KB
  GRU_seed42.keras                     238.2 KB
  GRU_seed5.keras                      238.2 KB
  GRU_seed7.keras                      238.2 KB
  GRU_seed77.keras                     238.2 KB
  TCN_seed0.keras                      412.1 KB
  TCN_seed1.keras                      412.1 KB
  TCN_seed2.keras                      412.1 KB
  TCN_seed42.keras                     412.1 KB
  TCN_seed7.keras                      412.1 KB
  feature_meta.json    

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# ── PER-SEED DEĞERLENDİRME (bağımsız; eğitim YOK, Cell 1 gerekmez) ──────────────
import os, glob, json, numpy as np, pandas as pd, tensorflow as tf
from joblib import load as joblib_load

OUTPUT_DIR="/content/drive/MyDrive/femto_rul/output"
CSV_DIR="/content/drive/MyDrive/femto_rul/processed_data"
CAP=125.0
META=['bearing','window_idx','time_s','rul_s','rul_min','t_star_s','cap_s','condition','split']
T8=['h_std','h_rms','h_kurtosis','h_peak','h_stft_band1_mean','h_stft_band0_mean','v_std','v_rms']

def phm_acc(a,p):
    er=100.0*(a-p)/(a+1e-10); ln=np.log(0.5)
    return np.exp(-ln*(er/5.0)) if er<=0 else np.exp(ln*(er/20.0))

def build(df, raw):
    out=[]
    for b,g in df.groupby('bearing'):
        g=g.sort_values('window_idx').copy()
        g['hi']=np.clip((g['time_s']-g['t_star_s'])/g['cap_s'],0,1)
        for c in raw:
            e=g[c].ewm(span=20,adjust=False).mean(); g[c]=e-e.iloc[:20].mean()
        for c in T8: g[c+'_trend']=g[c].diff(5).fillna(0)
        out.append(g)
    return pd.concat(out)

meta=json.load(open(f"{OUTPUT_DIR}/feature_meta.json")); TOP=meta["TOP"]; W=int(meta["W"])
sca=joblib_load(f"{OUTPUT_DIR}/feature_scaler.pkl")
te=pd.read_csv(f"{CSV_DIR}/features_test.csv")
RAW=[c for c in te.columns if c not in META]
teb=build(te,RAW)

def make_seq(g):
    g=g.sort_values('window_idx'); M=sca.transform(g[TOP].values); n=len(M)
    return np.stack([M[[max(0,i-j) for j in range(W-1,-1,-1)]] for i in range(n)])

def rul_from_hi(hi,times_s,t_star,gamma):
    h=np.clip(hi,1e-6,1)**gamma
    hs=pd.Series(h).ewm(span=20,adjust=False).mean().values; hl=max(hs[-1],1e-3)
    return float(np.clip(((times_s[-1]-t_star)/60.0)*(1-hl)/hl,0,CAP))

res=pd.read_csv(f"{OUTPUT_DIR}/track_b_results.csv").set_index("model")

for name in ["GRU","TCN"]:
    gs=float(res.loc[name,"gamma"]); phms=[]
    print(f"\n{name} per-seed (gamma={gs}):")
    for f in sorted(glob.glob(f"{OUTPUT_DIR}/{name}_seed*.keras")):
        m=tf.keras.models.load_model(f,compile=False); A,P=[],[]
        for b,g in teb.groupby('bearing'):
            g=g.sort_values('window_idx')
            hi=np.clip(m.predict(make_seq(g),verbose=0).ravel(),0,1)
            P.append(rul_from_hi(hi,g['time_s'].values,g['t_star_s'].iloc[0],gs))
            A.append(np.clip((g['time_s']+g['rul_s']).max()/60-g['time_s'].iloc[-1]/60,0,CAP))
        A,P=np.array(A),np.array(P); phm=np.mean([phm_acc(a,p) for a,p in zip(A,P)]); phms.append(phm)
        print(f"  {os.path.basename(f):20s} PHM={phm:.3f}  MAE={np.mean(np.abs(A-P)):.1f}  endpoint_r={np.corrcoef(A,P)[0,1]:+.3f}")
    phms=np.array(phms)
    print(f"  ÖZET (n={len(phms)}): PHM {phms.mean():.3f}±{phms.std():.3f}  [{phms.min():.3f}–{phms.max():.3f}]")


GRU per-seed (gamma=0.7):
  GRU_seed0.keras      PHM=0.212  MAE=28.8  endpoint_r=+0.625
  GRU_seed1.keras      PHM=0.213  MAE=28.9  endpoint_r=+0.631
  GRU_seed101.keras    PHM=0.274  MAE=31.9  endpoint_r=+0.670
  GRU_seed11.keras     PHM=0.228  MAE=29.7  endpoint_r=+0.621
  GRU_seed13.keras     PHM=0.157  MAE=37.3  endpoint_r=+0.431
  GRU_seed17.keras     PHM=0.143  MAE=37.5  endpoint_r=+0.543
  GRU_seed2.keras      PHM=0.184  MAE=34.6  endpoint_r=+0.539
  GRU_seed202.keras    PHM=0.218  MAE=30.2  endpoint_r=+0.642
  GRU_seed21.keras     PHM=0.308  MAE=25.6  endpoint_r=+0.685
  GRU_seed3.keras      PHM=0.238  MAE=28.7  endpoint_r=+0.630
  GRU_seed33.keras     PHM=0.213  MAE=29.8  endpoint_r=+0.632
  GRU_seed42.keras     PHM=0.239  MAE=28.0  endpoint_r=+0.662
  GRU_seed5.keras      PHM=0.228  MAE=30.8  endpoint_r=+0.643
  GRU_seed7.keras      PHM=0.154  MAE=35.6  endpoint_r=+0.484
  GRU_seed77.keras     PHM=0.098  MAE=39.0  endpoint_r=+0.482
  ÖZET (n=15): PHM 0.207±0.051  [0.098–0.30

In [ ]:
import glob, os, tensorflow as tf
FINAL={"GRU":[0,1,2,7,42], "TCN":[0,1,2,7,42], "rawTCN_v1":[0,1,2]}

def pack(paths):
    subs=[tf.keras.models.load_model(p,compile=False) for p in paths]
    for i,m in enumerate(subs): m._name=f"member_{i}"
    inp=tf.keras.Input(subs[0].input_shape[1:])
    outs=[m(inp) for m in subs]
    out=tf.keras.layers.Average()(outs) if len(outs)>1 else outs[0]
    return tf.keras.Model(inp,out,name="ensemble")

# 1) doğru seed'lerden TEK dosya ensemble
for tag,seeds in FINAL.items():
    paths=[f"{OUTPUT_DIR}/{tag}_seed{s}.keras" for s in seeds if os.path.exists(f"{OUTPUT_DIR}/{tag}_seed{s}.keras")]
    pack(paths).save(f"{OUTPUT_DIR}/{tag}_ensemble.keras")
    print(f"{tag}: {len(paths)} model → {tag}_ensemble.keras")

# 2) GRU'daki 15-seed artıklarını sil (sadece final-olmayanlar)
for tag,seeds in FINAL.items():
    keep={f"{tag}_seed{s}.keras" for s in seeds}
    for p in glob.glob(f"{OUTPUT_DIR}/{tag}_seed*.keras"):
        if os.path.basename(p) not in keep: os.remove(p); print("silindi:", os.path.basename(p))